# DropItRight — inspect an already-indexed song's stored profile

No extraction is run here. This just loads a song's profile JSON that `build_index.py` already computed and saved to `reference_db/profiles/<song_id>.json`, and prints every field: whole-song (lyrics, tonic, raga, MERT) + every segment's stored CAE / melodysim / lyric-slice.

Use this to sanity-check what actually got indexed, without paying extraction cost again.

In [ ]:
# Cell 1 -- config
import os, sys
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DB_DIR = "./reference_db"

import numpy as np
from reference_db import ReferenceDB

db = ReferenceDB(DB_DIR)
song_ids = db.list_song_ids()
print(f"{len(song_ids)} songs indexed:")
for sid in song_ids:
    print(f"  {sid}")

In [ ]:
# Cell 2 -- pick a song and load its stored profile (dict, jsonpickle-decoded)
SONG_ID = song_ids[0]  # <-- change to whichever song_id you want to inspect

profile = db.load_profile(SONG_ID)
print(f"loaded profile for: {SONG_ID}")
print(f"top-level keys: {sorted(profile.keys())}")

In [ ]:
# Cell 2b -- listen to the whole song
# song_id == the reference audio's filename without extension (see build_index.py),
# so we look it up back in the folder you originally indexed from.
import glob
from IPython.display import Audio

AUDIO_DIR = "tests/audio/reference_songs"  # <-- folder you ran build_index.py --audio-dir on

matches = glob.glob(os.path.join(AUDIO_DIR, SONG_ID + ".*"))
if not matches:
    print(f"no audio file found for song_id={SONG_ID!r} under {AUDIO_DIR} -- set AUDIO_DIR correctly")
else:
    song_audio_path = matches[0]
    print(f"playing: {song_audio_path}")
Audio(song_audio_path) if matches else None

## Whole-song / global fields

In [ ]:
# Cell 3 -- beat/rhythm + tonic + raga
print(f"title: {profile.get('title')}")
print(f"bpm: {profile.get('bpm')}")
print(f"rhythm (beats/bar): {profile.get('rhythm')}")
print(f"beat_track_source: {profile.get('beat_track_source')}")
print(f"downbeat_start: {profile.get('downbeat_start')}")
n_beats = len(profile.get('beat_times') or [])
print(f"n beat_times stored: {n_beats}")
print()
print(f"tonic_hz: {profile.get('tonic_hz')}")
print(f"raga: {profile.get('raga')}")
print(f"raga_confidence: {profile.get('raga_confidence')}")

In [ ]:
# Cell 4 -- MERT global embedding
mert = profile.get("global_mert_embedding")
if mert is not None:
    mert = np.asarray(mert)
    print(f"shape: {mert.shape}")
    print(f"norm: {np.linalg.norm(mert):.4f}")
    print(f"stats: min={mert.min():.4f} max={mert.max():.4f} mean={mert.mean():.4f}")
    print(f"first 10 dims: {mert[:10].round(4)}")
else:
    print("global_mert_embedding: None (MERT extraction failed for this song at index time)")

In [ ]:
# Cell 5 -- global lyrics transcript
lyrics = profile.get("global_lyrics")
if lyrics:
    print(f"full text:\n{lyrics.get('text')}\n")
    chunks = lyrics.get("chunks", [])
    print(f"n timestamped chunks: {len(chunks)}")
    for c in chunks[:15]:
        ts = c.get("timestamp", (None, None))
        print(f"  [{ts[0]}, {ts[1]}]  {c.get('text')!r}")
else:
    print("global_lyrics: None (ASR failed or produced nothing for this song)")

## Per-segment fields

`segments` is the list already computed and stored by `build_index.py` at indexing time -- CAE-Carnatic embedding, melodysim embedding, and the lyric slice cut from the global transcript, one entry per phrase-aligned segment at every scale (3/5/7/whole).

In [ ]:
# Cell 6 -- segment inventory grouped by duration_class
from itertools import groupby

segments = profile.get("segments") or []
segments_sorted = sorted(segments, key=lambda s: (str(s.get("duration_class")), s.get("start", 0)))

for duration_class, group in groupby(segments_sorted, key=lambda s: s.get("duration_class")):
    group = list(group)
    print(f"duration_class={duration_class}: {len(group)} segments")

In [ ]:
# Cell 7b -- listen to a specific stored segment (edit SEG_INDEX)
import librosa

SEG_INDEX = 0  # <-- index into segments_sorted
seg = segments_sorted[SEG_INDEX]
print(f"duration_class={seg.get('duration_class')}  [{seg.get('start'):.2f}s -> {seg.get('end'):.2f}s]")

y, sr = librosa.load(song_audio_path, sr=None, offset=seg["start"], duration=seg["end"] - seg["start"])
Audio(y, rate=sr)

In [ ]:
# Cell 7 -- full per-segment dump (CAE / melodysim / lyric slice), grouped by scale
for duration_class, group in groupby(segments_sorted, key=lambda s: s.get("duration_class")):
    group = list(group)
    print(f"\n=== duration_class={duration_class} ({len(group)} segments) ===")
    for seg in group:
        print(f"\n  [{seg.get('start'):.2f}s -> {seg.get('end'):.2f}s]  bars {seg.get('bar_start_idx')}->{seg.get('bar_end_idx')}")

        cae = seg.get("cae_embedding")
        if cae is not None:
            cae_arr = np.asarray(cae)
            print(f"    cae_embedding: shape={cae_arr.shape} norm={np.linalg.norm(cae_arr):.4f} first5={cae_arr[:5].round(4)}")
        else:
            print("    cae_embedding: None (failed at index time)")

        melodysim = seg.get("melodysim_embedding")
        print(f"    melodysim_embedding: {melodysim if melodysim is not None else 'None (stub not wired up yet)'}")

        print(f"    lyrics_slice: {seg.get('lyrics_slice')!r}")

## Summary across the whole DB (all indexed songs, not just SONG_ID)

In [ ]:
# Cell 8 -- coverage table across every indexed song
all_profiles = db.load_profiles(song_ids)

print(f"{'song_id':<40} {'mert':<6} {'tonic':<7} {'raga':<20} {'#seg':<6} {'cae_ok':<8} {'melodysim_ok':<13} {'lyrics_ok'}")
print("-" * 110)
for sid in song_ids:
    p = all_profiles[sid]
    segs = p.get("segments") or []
    n_cae_ok = sum(1 for s in segs if s.get("cae_embedding") is not None)
    n_melodysim_ok = sum(1 for s in segs if s.get("melodysim_embedding") is not None)
    n_lyrics_ok = sum(1 for s in segs if s.get("lyrics_slice"))
    print(f"{sid:<40} {'ok' if p.get('global_mert_embedding') else 'FAIL':<6} "
          f"{'ok' if p.get('tonic_hz') else 'FAIL':<7} {str(p.get('raga')):<20} {len(segs):<6} "
          f"{n_cae_ok}/{len(segs):<6} {n_melodysim_ok}/{len(segs):<11} {n_lyrics_ok}/{len(segs)}")

## Lyrics n-gram (BLEU-style) similarity sanity check

`fusion._lyrics_similarity` scores two lyric strings by n-gram overlap
(BLEU-style, symmetric, n capped to the shorter side's length). Try it
against real stored segment lyric slices -- both same-song pairs (should
score high/1.0 for identical text) and cross-song pairs (should score low
unless there's an actual shared line).

In [ ]:
# Cell 9 -- pairwise lyric-slice similarity matrix for this song's segments
from fusion import _lyrics_similarity

lyric_segments = [s for s in segments_sorted if s.get("lyrics_slice")]
print(f"{len(lyric_segments)} segments have non-empty lyric slices")

n = min(len(lyric_segments), 8)  # keep the printed matrix readable
sample = lyric_segments[:n]

print("\nsegments used:")
for i, s in enumerate(sample):
    print(f"  [{i}] duration_class={s['duration_class']} [{s['start']:.2f}-{s['end']:.2f}s]: {s['lyrics_slice']!r}")

print("\nsimilarity matrix:")
header = "      " + "".join(f"{i:>7}" for i in range(n))
print(header)
for i, a in enumerate(sample):
    row = f"[{i:>2}]  "
    for j, b in enumerate(sample):
        score = _lyrics_similarity(a["lyrics_slice"], b["lyrics_slice"])
        row += f"{score:7.3f}"
    print(row)